# WSI Inference v5.1 — fast (patched easy-win edition)

Same model + outputs as `infer_wsi_v5_fast.ipynb`, with these fixes from `V5_FIRST_RUN_REPORT.md` and `LOAD_TO_RAM_ASSESSMENT.md`:

- **Patch A — Forward 20× speedup** — bulk D2H (3 syncs/batch instead of 384), drop autocast on pre-halved model, drop per-batch `pin_memory()`, apply class permutation once on GPU.
- **Patch B — Tissue mask actually works** — Otsu on V channel (was fixed thresholds) + coverage safety net + bbox clipped to slide bounds.
- **Patch C — Kill polygon dedup** — `DEDUP_USE_POLYGON_OVERLAP=False` + `DEDUP_MIN_DIST_PX=12.0` (absorbs the 0.3% that polygon dedup caught for 49 s/slide).
- **Patch D — Smoke-test flag** — `SMOKE_TEST_MAX_TILES` for fast iteration.
- **Patch E — `LOAD_TO_RAM=False` default + auto-pick** — RAM mode only for tiny local slides; disk-backed DataLoader for everything else.
- **Patch F — `OPENSLIDE_CACHE_SIZE=128 MB`** — set before openslide import for free 64-px overlap-region cache hits.

**Predicted wall-clock** (slide 1, 38400×26752 px, ~28k tiles): **503 s → ~170 s** (~3× end-to-end).

**Run cells top-to-bottom. Edit cell 2 (Parameters).**

| Cell | Purpose |
|------|---------|
| 1 | Imports (sets `OPENSLIDE_CACHE_SIZE` before openslide) |
| **2** | **Parameters** |
| 3 | Load model |
| 4 | v5.1 helpers inline (Otsu mask, bbox clip, bulk D2H forward, streaming writer) |
| 5 | Colour helpers |
| 6 | Diagnostic — open slide + tissue mask preview + coverage report |
| 7 | Diagnostic — single-tile inference |
| **8** | **Batch inference — auto-picks LOAD_TO_RAM per slide** |
| 8b | Combined summary |
| 9 | Results bar chart |


---
## 1 · Imports

Sets `OPENSLIDE_CACHE_SIZE` BEFORE openslide is imported so workers inherit it.


In [ ]:
import os
# Patch F: bump openslide tile cache before openslide gets loaded.
os.environ.setdefault('OPENSLIDE_CACHE_SIZE', str(128 * 1024 * 1024))   # 128 MB / handle

import gzip
import io
import json
import math
import time
import cProfile
import pstats
import concurrent.futures
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import BoundaryNorm, ListedColormap
from tqdm import tqdm
import torch
import openslide
import yaml

try:
    import psutil
    _HAS_PSUTIL = True
except ImportError:
    _HAS_PSUTIL = False

from inference_utils import *
from geometry import *

try:
    import cv2
    _HAS_CV2 = True
except ImportError:
    _HAS_CV2 = False
    print('[warn] opencv-python not installed; falling back to skimage for tissue mask.')
try:
    import orjson  # noqa: F401
    _HAS_ORJSON = True
except ImportError:
    _HAS_ORJSON = False
    print('[warn] orjson not installed (pip install orjson) — using stdlib json fallback.')

print(f'Imports OK   |   opencv={_HAS_CV2}   orjson={_HAS_ORJSON}   psutil={_HAS_PSUTIL}')
print(f'OPENSLIDE_CACHE_SIZE={os.environ["OPENSLIDE_CACHE_SIZE"]} bytes')


---
## 2 · Parameters


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
WSI_DIR      = Path(r'\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40')
WSI_GLOB     = '*.ndpi'
SLIDE_INDEX  = 1   # diagnostic cells only; batch processes ALL matches

_cands = sorted(WSI_DIR.glob(WSI_GLOB))
if not _cands:
    raise FileNotFoundError(f'No files matching {WSI_GLOB!r} under {WSI_DIR}')
SLIDE_PATH = _cands[int(SLIDE_INDEX) % len(_cands)]

WEIGHTS_PATH = Path(r'\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\cellvit_training\dataset_256_40k_48_slides\convnext_stardist_multitask_runs\run_gs40_gs55_v2\convnext_stardist_mt_gs40_gs55_v2_cls2L_skip128\best.pt')
CONFIG_PATH  = Path(r'C:\Users\Andre\cursor_projects\Convnext_stardist\shared_convnext_stardist_decoder\config_gs40_gs55_multitask.yaml')
OUT_DIR      = Path(r'\\kittyserverdw\Andre_kit\data\students\Diogo\data\fetal\GS40\config_gs40_gs55_multitask_v51_fast')

# ── Class display palette ──────────────────────────────────────────────────────
LABELS_VIZ = [
    'bone',   'brain',  'eye',       'heart',     'lungs',
    'GI',     'liver',  'spleen',    'pancreas',  'kidney',
    'mesokidney', 'collagen', 'ear', 'nontissue', 'thymus',
    'thyroid', 'bladder', 'skull',   'spleen2',
]
COLORS_VIZ = [
    [214,212,161], [247,184, 67], [136,232, 95], [140, 13, 13], [ 38, 27,166],
    [ 13,125, 11], [179, 50,108], [228,235,131], [156, 96,235], [ 46,190,230],
    [150,255,245], [254,222,255], [235,154,108], [255,255,255], [  9, 64,116],
    [255,255, 74], [178,178,  0], [214,212,161], [ 54, 83, 89],
]
PERMUTE_LEGACY_TO_DISPLAY = False
LEGACY_CLASS_ORDER = [
    'bladder', 'bone',   'brain',      'collagen',  'ear',
    'eye',     'gi',     'heart',      'kidney',    'liver',
    'lungs',   'mesokidney', 'nontissue', 'pancreas', 'skull',
    'spleen',  'spleen2', 'thymus',    'thyroid',
]

# ── Slide reading ──────────────────────────────────────────────────────────────
SLIDE_LEVEL  = 0
# Patch E: LOAD_TO_RAM defaults to False; auto-pick overrides per slide
LOAD_TO_RAM      = False
AUTO_LOAD_MODE   = True   # auto-decide LOAD_TO_RAM per slide based on size + RAM + network
AUTO_RAM_FRAC    = 0.40   # only use RAM mode if slide < frac * available RAM
AUTO_RAM_MAX_GB  = 1.5    # AND slide raw < this many GB

# ── Tile geometry ──────────────────────────────────────────────────────────────
TILE_SIZE    = 256
TILE_OVERLAP = 64
VALID_MARGIN = 32     # = TILE_OVERLAP / 2 (correct for stride 192)

# ── Detection thresholds ───────────────────────────────────────────────────────
PROB_THRESH  = 0.45
NMS_DIST     = 8

# ── Deduplication ─────────────────────────────────────────────────────────────
# Patch C: raise centroid threshold; disable polygon dedup
DEDUP_MIN_DIST_PX         = 12.0   # was 10.0 — absorbs 99%+ of polygon-dedup cases
DEDUP_USE_POLYGON_OVERLAP = False  # was True — saved ~49 s/slide for 0.3% removal
DEDUP_OVERLAP_RATIO       = 0.40
DEDUP_MIN_IOU             = 0.28
DEDUP_OVERLAP_GRID_PX     = 32.0

# ── Peak refinement ────────────────────────────────────────────────────────────
REFINE_PEAK_LOCAL_COM = True
REFINE_PEAK_RADIUS    = 8

# ── Hardware / speed ──────────────────────────────────────────────────────────
DEVICE_STR        = 'cuda'
USE_FP16          = True       # ignored if USE_BF16=True
USE_BF16          = False      # bf16 has fp32 range → safer for softplus on head_dist
USE_CHANNELS_LAST = False      # only enable with bf16/fp16 once validated
NUM_WORKERS       = 8
BATCH_TILES       = 128
PREFETCH          = 2

# ── Tissue ROI mask (Patch B: Otsu on V) ───────────────────────────────────────
TISSUE_MASK_ENABLED       = True
TISSUE_MASK_THUMB_W       = 2048
TISSUE_MASK_USE_OTSU      = True   # NEW — was False (fixed thresholds) in v5
TISSUE_MASK_VAL_THRESH    = 240    # only used when USE_OTSU=False
TISSUE_MASK_SAT_THRESH    = 15     # additive saturation guard (Otsu | S>this)
TISSUE_MASK_MIN_COMP_FRAC = 5e-4   # was 1e-4 in v5 — strips JPEG speckle
TISSUE_MASK_DILATE_TILES  = 1
TISSUE_TILE_KEEP_FRAC     = 0.01
TISSUE_BBOX_CROP          = True
TISSUE_MASK_CACHE         = True
TISSUE_COVERAGE_HIGH      = 0.95   # NEW — disable filter if mask covers > this
TISSUE_COVERAGE_LOW       = 0.02   # NEW — warn if mask covers < this

# ── GeoJSON export ─────────────────────────────────────────────────────────────
GEOJSON_COORD_INT     = True
GEOJSON_GZIP          = False
INCLUDE_CLASS_PROBS   = False
GEOJSON_FLUSH_BYTES   = 1 << 20

# ── Diagnostic ─────────────────────────────────────────────────────────────────
SAMPLE_MODE  = 'random'
SAMPLE_X     = 0
SAMPLE_Y     = 0

# ── Profiling / smoke test ─────────────────────────────────────────────────────
RUN_CPROFILE_FIRST_SLIDE = False
SMOKE_TEST_MAX_TILES     = 0   # >0 to limit each slide to N tiles for quick iteration

# ── Derived ────────────────────────────────────────────────────────────────────
DEVICE = torch.device(DEVICE_STR if torch.cuda.is_available() else 'cpu')
OUT_DIR.mkdir(parents=True, exist_ok=True)
if USE_BF16 and USE_FP16:
    print('[info] USE_BF16=True overrides USE_FP16=True')
    USE_FP16 = False
if USE_CHANNELS_LAST and not (USE_FP16 or USE_BF16):
    print('[warn] USE_CHANNELS_LAST is intended for fp16/bf16; enabling fp16.')
    USE_FP16 = True

print(f'Device      : {DEVICE}')
print(f'Slide       : {SLIDE_PATH.name}')
print(f'Output      : {OUT_DIR}')
print(f'Precision   : {"bf16" if USE_BF16 else "fp16" if USE_FP16 else "fp32"}'
      + ('  channels_last' if USE_CHANNELS_LAST else ''))
print(f'DataLoader  : workers={NUM_WORKERS}  prefetch={PREFETCH}  batch={BATCH_TILES}')
print(f'LOAD_TO_RAM : {LOAD_TO_RAM}  (auto-pick={AUTO_LOAD_MODE})')
print(f'Tissue mask : {"ON" if TISSUE_MASK_ENABLED else "OFF"}'
      f'  ({"Otsu" if TISSUE_MASK_USE_OTSU else f"V<{TISSUE_MASK_VAL_THRESH}"} | S>{TISSUE_MASK_SAT_THRESH})')
print(f'Dedup       : centroid {DEDUP_MIN_DIST_PX}px '
      f'+ {"polygon" if DEDUP_USE_POLYGON_OVERLAP else "NO polygon"}')
if SMOKE_TEST_MAX_TILES > 0:
    print(f'SMOKE TEST  : limiting each slide to {SMOKE_TEST_MAX_TILES} tiles')


---
## 3 · Load model


In [ ]:
model, idx2label = load_model_and_classes(WEIGHTS_PATH, CONFIG_PATH, DEVICE)
print(f'Model     : {type(model).__name__}  ({sum(p.numel() for p in model.parameters())/1e6:.1f} M params)')
print(f'n_rays    : {model.n_rays}   num_classes : {model.num_classes}')

cls_perm = None
if PERMUTE_LEGACY_TO_DISPLAY:
    cls_perm, idx2label = build_class_permutation(CONFIG_PATH, LEGACY_CLASS_ORDER)
    print(f'\nPermutation ON  →  first 6: {cls_perm[:6].tolist()}')
else:
    print('\nPermutation OFF — checkpoint uses config class_names order directly.')

if USE_BF16:
    model = model.to(dtype=torch.bfloat16)
    print('Model dtype : bfloat16')
elif USE_FP16:
    model = model.to(dtype=torch.float16)
    print('Model dtype : float16')
else:
    model = model.to(dtype=torch.float32)
    print('Model dtype : float32')
if USE_CHANNELS_LAST:
    model = model.to(memory_format=torch.channels_last)
    print('Memory format: channels_last')
model.eval()

# Patch A: pre-build the cls_perm CUDA index tensor once if needed
_CLS_PERM_GPU = None
if cls_perm is not None:
    _CLS_PERM_GPU = torch.as_tensor(cls_perm, device=DEVICE, dtype=torch.long)
    print(f'cls_perm staged on {DEVICE} as long index tensor')


---
## 4 · v5.1 helpers (inline)

All four patched helpers in one cell so you can iterate fast:

- **Patch B**: `compute_tissue_mask_thumbnail` — Otsu on V channel by default. `tissue_bbox_level0` clips to slide bounds.
- **Patch A**: `batch_forward_v51` — bulk D2H, no autocast, no per-batch pinning, GPU-side `cls_perm`.
- `process_tile_batch_v51` — same as v5 but routed through `batch_forward_v51`.
- `write_geojson_streaming` — unchanged from v5 (already worked).


In [ ]:
# ── 4.1 Tissue mask (Patch B: Otsu + bbox clip + safety net) ─────────────────
def compute_tissue_mask_thumbnail(
    slide,
    *,
    target_thumbnail_w: int = 2048,
    use_otsu: bool = True,
    val_thresh: int = 240,
    sat_thresh: int = 15,
    min_component_frac: float = 5e-4,
    dilate_tiles: int = 1,
    tile_size_l0: int = 256,
):
    """Thumbnail tissue mask. Returns (mask_bool[H_t, W_t], scale=level0_px/thumb_px).
    Patch B: Otsu on V by default (auto-calibrates per slide). Saturation guard is
    additive only — never subtracts tissue. Morphological close + small-component
    removal + tile-sized safety dilation."""
    W0, H0 = slide.level_dimensions[0]
    scale = max(1.0, W0 / float(target_thumbnail_w))
    tw, th = int(round(W0 / scale)), int(round(H0 / scale))
    thumb = np.asarray(slide.get_thumbnail((tw, th)).convert('RGB'), dtype=np.uint8)

    if _HAS_CV2:
        hsv = cv2.cvtColor(thumb, cv2.COLOR_RGB2HSV)
        S, V = hsv[..., 1], hsv[..., 2]
        if use_otsu:
            # Otsu on V channel — auto-calibrates per slide
            _, m_v = cv2.threshold(V, 0, 1, cv2.THRESH_BINARY_INV | cv2.THRESH_OTSU)
        else:
            m_v = (V < val_thresh).astype(np.uint8)
        # Saturation guard catches faint eosin / pink stained regions Otsu may miss.
        # OR (additive only — extends tissue, never subtracts)
        m_s = (S > sat_thresh).astype(np.uint8)
        mask = (m_v.astype(np.uint8) | m_s).astype(np.uint8)
        # Morphological close (fill small holes inside tissue)
        k_close = max(3, int(round(tile_size_l0 / scale / 4)) | 1)
        mask = cv2.morphologyEx(
            mask, cv2.MORPH_CLOSE,
            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close)),
        )
        # Drop dust/specks below min_component_frac of slide area
        n, lab, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        keep = stats[:, cv2.CC_STAT_AREA] >= max(4, int(mask.size * min_component_frac))
        keep[0] = False
        mask = keep[lab].astype(np.uint8)
        # Safety dilation by ~1 tile so edge tissue is never dropped
        if dilate_tiles > 0:
            k_dil = max(3, int(round(dilate_tiles * tile_size_l0 / scale)) | 1)
            mask = cv2.dilate(
                mask,
                cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_dil, k_dil)),
            )
    else:
        # skimage fallback
        from skimage.color import rgb2hsv
        from skimage.filters import threshold_otsu
        from skimage.morphology import (
            binary_closing, disk, remove_small_objects, binary_dilation,
        )
        hsv = rgb2hsv(thumb)
        V = (hsv[..., 2] * 255).astype(np.uint8)
        S = (hsv[..., 1] * 255).astype(np.uint8)
        if use_otsu:
            t_v = int(threshold_otsu(V))
            m_v = V < t_v
        else:
            m_v = V < val_thresh
        m_s = S > sat_thresh
        mask = m_v | m_s
        k_close = max(3, int(round(tile_size_l0 / scale / 4)))
        mask = binary_closing(mask, disk(k_close // 2))
        mask = remove_small_objects(mask, max(4, int(mask.size * min_component_frac)))
        if dilate_tiles > 0:
            k_dil = max(3, int(round(dilate_tiles * tile_size_l0 / scale)))
            mask = binary_dilation(mask, disk(k_dil // 2))
    return mask.astype(bool), float(scale)


def filter_tile_coords_by_mask(tile_coords, mask, mask_scale, *, frac: float = 0.01):
    Hm, Wm = mask.shape
    kept = []
    for (x0, y0, tw, th) in tile_coords:
        r0 = max(0, int(np.floor(y0 / mask_scale)))
        r1 = min(Hm, int(np.ceil((y0 + th) / mask_scale)) + 1)
        c0 = max(0, int(np.floor(x0 / mask_scale)))
        c1 = min(Wm, int(np.ceil((x0 + tw) / mask_scale)) + 1)
        if r1 > r0 and c1 > c0 and float(mask[r0:r1, c0:c1].mean()) >= frac:
            kept.append((x0, y0, tw, th))
    return kept, len(tile_coords) - len(kept)


def tissue_bbox_level0(mask, mask_scale, slide_dims, pad_px: int = 256):
    """Patch B: clip bbox to slide dimensions so we never read beyond the slide."""
    W0, H0 = slide_dims
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, 0, 0
    x0 = max(0, int(np.floor(xs.min() * mask_scale)) - pad_px)
    y0 = max(0, int(np.floor(ys.min() * mask_scale)) - pad_px)
    x1 = min(W0, int(np.ceil((xs.max() + 1) * mask_scale)) + pad_px)
    y1 = min(H0, int(np.ceil((ys.max() + 1) * mask_scale)) + pad_px)
    return x0, y0, x1 - x0, y1 - y0


def load_or_build_tissue_mask(slide, slide_path, out_dir, *, cache: bool = True, **kw):
    cache_path = Path(out_dir) / f'{Path(slide_path).stem}_tissue_mask.npz'
    if cache and cache_path.exists():
        d = np.load(cache_path)
        return d['mask'].astype(bool), float(d['scale'])
    mask, scale = compute_tissue_mask_thumbnail(slide, **kw)
    if cache:
        np.savez_compressed(cache_path, mask=mask.astype(np.uint8), scale=np.float32(scale))
    return mask, scale


# ── 4.2 GPU-side normalize forward (Patch A: bulk D2H, no autocast, no pin) ──
_IMNET_MEAN = (0.485, 0.456, 0.406)
_IMNET_STD  = (0.229, 0.224, 0.225)

@torch.inference_mode()
def batch_forward_v51(
    model,
    patches_hwc,
    device,
    *,
    fp16: bool = False,
    bf16: bool = False,
    channels_last: bool = False,
    cls_perm_gpu=None,
):
    """Patch A: bulk D2H (3 syncs/batch), no autocast on pre-halved model,
    no per-batch pin_memory(), apply cls_perm once on GPU.
    Inputs: list of uint8 HWC numpy arrays.
    Returns: list of (prob, dist_map, cls_log) numpy arrays per tile."""
    if bf16:
        infer_dtype = torch.bfloat16
    elif fp16 and device.type == 'cuda':
        infer_dtype = torch.float16
    else:
        infer_dtype = torch.float32

    sizes = [(p.shape[0], p.shape[1]) for p in patches_hwc]
    max_h = int(math.ceil(max(s[0] for s in sizes) / 32) * 32)
    max_w = int(math.ceil(max(s[1] for s in sizes) / 32) * 32)

    arrs = []
    for p in patches_hwc:
        ph, pw = p.shape[:2]
        if ph != max_h or pw != max_w:
            pad = np.zeros((max_h, max_w, 3), dtype=np.uint8)
            pad[:ph, :pw] = p
            p = pad
        arrs.append(p)

    # Patch A: NO pin_memory() per batch — single-shot pinning is slower than skipping
    batch_u8 = torch.from_numpy(np.stack(arrs))                 # (N, H, W, 3) uint8
    batch = batch_u8.to(device, non_blocking=True).permute(0, 3, 1, 2).contiguous()
    batch = batch.to(infer_dtype).div_(255.0)
    mean = torch.tensor(_IMNET_MEAN, device=device, dtype=infer_dtype).view(1, 3, 1, 1)
    std  = torch.tensor(_IMNET_STD,  device=device, dtype=infer_dtype).view(1, 3, 1, 1)
    batch = (batch - mean) / std
    if channels_last:
        batch = batch.contiguous(memory_format=torch.channels_last)

    # Patch A: NO autocast — model is already pre-halved; autocast would fight it
    prob_logit, dist_b, cls_b = model(batch)

    # Patch A: apply cls_perm ONCE on GPU (was per-tile numpy fancy index in v5)
    if cls_perm_gpu is not None:
        cls_b = cls_b.index_select(1, cls_perm_gpu)

    # Patch A: 3 BULK D2H transfers (3 syncs total, not 384 like v5)
    prob_all = torch.sigmoid(prob_logit).cpu().float().numpy()   # (N, 1, H, W)
    dist_all = dist_b.contiguous().cpu().float().numpy()         # (N, 32, H, W)
    cls_all  = cls_b.contiguous().cpu().float().numpy()          # (N, 19, H, W)

    return [
        (prob_all[i, 0, :ph, :pw],
         dist_all[i, :, :ph, :pw],
         cls_all[i, :, :ph, :pw])
        for i, (ph, pw) in enumerate(sizes)
    ]


# ── 4.3 process_tile_batch_v51 (routes through Patch A forward) ───────────────
def process_tile_batch_v51(
    meta,
    patches,
    model,
    device,
    *,
    fp16: bool = False,
    bf16: bool = False,
    channels_last: bool = False,
    cls_perm_gpu=None,
    nms_dist: int = 8,
    prob_thresh: float = 0.45,
    refine_local_com: bool = True,
    refine_radius_px: int = 8,
    valid_margin: int = 0,
    w_slide: int,
    h_slide: int,
    idx2label,
    color_for_name=None,
    use_fast_vote_class: bool = True,
    vote_window_px: int = 9,
    coord_int: bool = True,
    include_class_probs: bool = False,
):
    """v5.1: integer-pixel rings + colorRGB inline; routed through batch_forward_v51."""
    from geometry import (
        local_peaks,
        dists_and_coords_from_peaks,
        polygon_ring_rowcol,
        vote_class as poly_vote_class,
    )

    _t = time.perf_counter()
    results = batch_forward_v51(
        model, patches, device,
        fp16=fp16, bf16=bf16, channels_last=channels_last,
        cls_perm_gpu=cls_perm_gpu,
    )
    gpu_time = time.perf_counter() - _t

    vm = int(valid_margin)
    hw = max(1, vote_window_px // 2)
    features = []

    for (x0, y0, tw, th), (prob, dist_m, cls_l) in zip(meta, results):
        pks = local_peaks(prob, min_distance=int(nms_dist), thresh=float(prob_thresh))
        if not len(pks):
            continue
        _, coords = dists_and_coords_from_peaks(
            dist_m, pks, prob,
            refine_local_com=refine_local_com,
            refine_radius_px=int(refine_radius_px),
        )
        _left, _top    = (x0 == 0), (y0 == 0)
        _right, _bottom = (x0 + tw >= w_slide), (y0 + th >= h_slide)
        H_t, W_t = prob.shape
        for k in range(coords.shape[0]):
            pr, pc = int(pks[k, 0]), int(pks[k, 1])
            if vm:
                if not _left   and pc < vm:        continue
                if not _right  and pc >= tw - vm:  continue
                if not _top    and pr < vm:         continue
                if not _bottom and pr >= th - vm:   continue
            if use_fast_vote_class:
                r0 = max(0, pr - hw); r1 = min(H_t, pr + hw)
                c0 = max(0, pc - hw); c1 = min(W_t, pc + hw)
                lw = cls_l[:, r0:r1, c0:c1].mean(axis=(1, 2))
                lw = lw - lw.max()
                e = np.exp(lw)
                probs_k = (e / (e.sum() + 1e-10)).astype(np.float32)
                cls_id = int(probs_k.argmax())
            else:
                cls_id, probs_k = poly_vote_class(cls_l, coords[k], (th, tw))
            ring = polygon_ring_rowcol(coords[k]) + np.array([x0, y0], dtype=np.float32)
            if coord_int:
                ring = np.rint(ring).astype(np.int32)
            name = idx2label.get(cls_id, f'class_{cls_id}')
            r_, g_, b_ = color_for_name(name) if color_for_name else (128, 128, 128)
            color_rgb = (int(r_) << 16) | (int(g_) << 8) | int(b_)
            props = {
                'classification': {
                    'name': name,
                    'index': int(cls_id),
                    'colorRGB': int(color_rgb),
                },
                'prob_peak': float(prob[pr, pc]),
            }
            if include_class_probs:
                props['class_probs'] = {
                    idx2label.get(i, str(i)): float(p)
                    for i, p in enumerate(probs_k)
                }
            features.append({
                'type': 'Feature',
                'geometry': {'type': 'Polygon', 'coordinates': [ring]},
                'properties': props,
            })
    return features, gpu_time


# ── 4.4 Streaming GeoJSON writer (unchanged from v5) ─────────────────────────
def write_geojson_streaming(
    features_iter,
    path,
    *,
    gzip_compress: bool = False,
    flush_bytes: int = 1 << 20,
):
    opener = gzip.open if gzip_compress else open
    use_orjson = _HAS_ORJSON
    if use_orjson:
        opt = orjson.OPT_SERIALIZE_NUMPY
    n_written = 0
    with opener(str(path), 'wb') as f:
        f.write(b'{"type":"FeatureCollection","features":[')
        first = True
        buf = bytearray()
        for feat in features_iter:
            if first:
                first = False
            else:
                buf.append(0x2C)
            if use_orjson:
                buf += orjson.dumps(feat, option=opt)
            else:
                geom = feat.get('geometry', {})
                if geom.get('type') == 'Polygon':
                    coords = geom.get('coordinates', [])
                    coords_py = [
                        ring.tolist() if isinstance(ring, np.ndarray) else ring
                        for ring in coords
                    ]
                    feat = {**feat, 'geometry': {'type': 'Polygon', 'coordinates': coords_py}}
                buf += json.dumps(feat, separators=(',', ':')).encode('utf-8')
            n_written += 1
            if len(buf) > flush_bytes:
                f.write(buf); buf.clear()
        if buf:
            f.write(buf)
        f.write(b']}')
    return n_written


# ── 4.5 Auto LOAD_TO_RAM picker (Patch E) ────────────────────────────────────
def auto_load_to_ram(
    slide_path,
    slide,
    level: int,
    *,
    max_gb: float = 1.5,
    ram_frac: float = 0.40,
) -> tuple[bool, str]:
    """Decide LOAD_TO_RAM per-slide. Returns (decision, reason).
    Conservative: defaults to False (DataLoader mode) unless tiny + local + fits."""
    W, H = slide.level_dimensions[level]
    raw_bytes = W * H * 3
    raw_gb = raw_bytes / 1e9
    is_network = (
        str(slide_path).startswith(r'\\') or '://' in str(slide_path)
    )
    avail_gb = (psutil.virtual_memory().available / 1e9) if _HAS_PSUTIL else 8.0
    fits = raw_bytes < ram_frac * (avail_gb * 1e9)
    small = raw_gb < max_gb
    if is_network:
        return False, f'network share (raw={raw_gb:.1f} GB)'
    if not small:
        return False, f'too big (raw={raw_gb:.1f} GB > {max_gb} GB)'
    if not fits:
        return False, f'tight RAM (raw={raw_gb:.1f} GB, avail={avail_gb:.1f} GB)'
    return True, f'small + local + fits (raw={raw_gb:.1f} GB, avail={avail_gb:.1f} GB)'


print('v5.1 helpers loaded:')
print('  • compute_tissue_mask_thumbnail   — Otsu on V (Patch B)')
print('  • tissue_bbox_level0              — clipped to slide bounds (Patch B)')
print('  • batch_forward_v51               — bulk D2H, no autocast, no pin (Patch A)')
print('  • process_tile_batch_v51          — uses batch_forward_v51')
print('  • write_geojson_streaming         — orjson + numpy + 1 MB flush')
print('  • auto_load_to_ram                — per-slide RAM/disk decision (Patch E)')


---
## 5 · Colour helpers


In [ ]:
_cm = build_class_colormap(model.num_classes, idx2label, LABELS_VIZ, COLORS_VIZ)
_color_for_class_idx  = _cm['color_for_idx']
_color_for_class_name = _cm['color_for_name']
CMAP_CLASS = _cm['cmap']
NORM_CLASS = _cm['norm']
_class_colors_float = _cm['colors_float']

fig, ax = plt.subplots(figsize=(12, 0.6))
ax.imshow([_class_colors_float], aspect='auto')
ax.set_xticks(range(model.num_classes))
ax.set_xticklabels([idx2label.get(i, str(i)) for i in range(model.num_classes)],
                   rotation=45, ha='right', fontsize=8)
ax.set_yticks([])
ax.set_title('Class palette')
plt.tight_layout()
plt.show()


---
## 6 · Diagnostic — open slide + tissue mask preview

Verifies the **Otsu** mask on `SLIDE_PATH`. Prints coverage statistics so you can spot a misbehaving mask before the batch run.


In [ ]:
slide = openslide.open_slide(str(SLIDE_PATH))
W_slide, H_slide = slide.level_dimensions[SLIDE_LEVEL]
print(f'Slide : {SLIDE_PATH.name}  {W_slide}×{H_slide} px at level {SLIDE_LEVEL}')

if TISSUE_MASK_ENABLED:
    _t = time.perf_counter()
    tissue_mask, mask_scale = compute_tissue_mask_thumbnail(
        slide,
        target_thumbnail_w=TISSUE_MASK_THUMB_W,
        use_otsu=TISSUE_MASK_USE_OTSU,
        val_thresh=TISSUE_MASK_VAL_THRESH,
        sat_thresh=TISSUE_MASK_SAT_THRESH,
        min_component_frac=TISSUE_MASK_MIN_COMP_FRAC,
        dilate_tiles=TISSUE_MASK_DILATE_TILES,
        tile_size_l0=TILE_SIZE,
    )
    _dt = time.perf_counter() - _t
    coverage = float(tissue_mask.mean())
    print(f'Tissue mask : {tissue_mask.shape}  scale={mask_scale:.1f}  '
          f'coverage={coverage:.1%}  ({_dt*1000:.0f} ms)')
    if coverage > TISSUE_COVERAGE_HIGH:
        print(f'  WARN: coverage > {TISSUE_COVERAGE_HIGH:.0%} — mask will be auto-disabled in batch loop')
    elif coverage < TISSUE_COVERAGE_LOW:
        print(f'  WARN: coverage < {TISSUE_COVERAGE_LOW:.0%} — possibly mis-thresholded; verify thumbnail')
    bx, by, bw, bh = tissue_bbox_level0(
        tissue_mask, mask_scale, (W_slide, H_slide), pad_px=TILE_SIZE,
    )
    full_area = W_slide * H_slide
    bbox_area = max(1, bw * bh)
    print(f'Tissue bbox @ level 0 : ({bx},{by})  {bw}×{bh}   '
          f'(bbox = {100*bbox_area/full_area:.1f}% of slide area)')
    _all_coords = build_tile_coords(W_slide, H_slide, TILE_SIZE, TILE_OVERLAP)
    _kept, _drop = filter_tile_coords_by_mask(
        _all_coords, tissue_mask, mask_scale, frac=TISSUE_TILE_KEEP_FRAC,
    )
    print(f'Tiles  : {len(_all_coords):,} total  →  {len(_kept):,} kept  '
          f'({100*_drop/max(len(_all_coords),1):.1f}% skipped)')
else:
    tissue_mask = None
    mask_scale = 1.0
    bx, by, bw, bh = 0, 0, W_slide, H_slide
    coverage = 1.0
    _all_coords = build_tile_coords(W_slide, H_slide, TILE_SIZE, TILE_OVERLAP)
    _kept = _all_coords

# Diagnostic tile
sx, sy, rw, rh = pick_diagnostic_tile(
    W_slide, H_slide, TILE_SIZE,
    mode=SAMPLE_MODE, sample_x=SAMPLE_X, sample_y=SAMPLE_Y,
)
sample_rgb = np.asarray(
    slide.read_region(
        (int(sx * slide.level_downsamples[SLIDE_LEVEL]),
         int(sy * slide.level_downsamples[SLIDE_LEVEL])),
        SLIDE_LEVEL, (rw, rh),
    ).convert('RGB'),
    dtype=np.uint8,
)

thumb_w = 800
thumb_h = int(H_slide * thumb_w / W_slide)
thumb = np.asarray(slide.get_thumbnail((thumb_w, thumb_h)).convert('RGB'))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(thumb)
axes[0].set_title(f'Slide thumbnail ({SLIDE_PATH.name})')
axes[0].axis('off')
sx_t = int(sx * thumb_w / W_slide); sy_t = int(sy * thumb_h / H_slide)
rw_t = max(3, int(rw * thumb_w / W_slide)); rh_t = max(3, int(rh * thumb_h / H_slide))
axes[0].add_patch(mpatches.Rectangle((sx_t, sy_t), rw_t, rh_t,
                                     linewidth=2, edgecolor='lime', facecolor='none',
                                     label='diagnostic tile'))
if TISSUE_MASK_ENABLED and TISSUE_BBOX_CROP and bw > 0 and bh > 0:
    bx_t = int(bx * thumb_w / W_slide); by_t = int(by * thumb_h / H_slide)
    bw_t = int(bw * thumb_w / W_slide); bh_t = int(bh * thumb_h / H_slide)
    axes[0].add_patch(mpatches.Rectangle((bx_t, by_t), bw_t, bh_t,
                                         linewidth=2, edgecolor='cyan', facecolor='none',
                                         label='tissue bbox'))
axes[0].legend(loc='upper right', fontsize=8)
if TISSUE_MASK_ENABLED:
    if _HAS_CV2:
        mask_resized = cv2.resize(
            tissue_mask.astype(np.uint8) * 255, (thumb_w, thumb_h),
            interpolation=cv2.INTER_NEAREST,
        ).astype(bool)
    else:
        from skimage.transform import resize as _sk_resize
        mask_resized = _sk_resize(
            tissue_mask.astype(np.uint8), (thumb_h, thumb_w),
            order=0, preserve_range=True,
        ).astype(bool)
    overlay = thumb.copy()
    overlay[~mask_resized] = (overlay[~mask_resized] * 0.3).astype(np.uint8)
    axes[1].imshow(overlay)
    axes[1].set_title(f'Mask overlay (kept {len(_kept):,}/{len(_all_coords):,} tiles, cov={coverage:.0%})')
    axes[1].axis('off')
    axes[2].imshow(tissue_mask, cmap='gray')
    method = 'Otsu V' if TISSUE_MASK_USE_OTSU else f'V<{TISSUE_MASK_VAL_THRESH}'
    axes[2].set_title(f'Tissue mask ({method} | S>{TISSUE_MASK_SAT_THRESH})')
    axes[2].axis('off')
    plt.savefig(OUT_DIR / f'{SLIDE_PATH.stem}_tissue_mask.png', dpi=120, bbox_inches='tight')
else:
    axes[1].imshow(sample_rgb); axes[1].set_title('Diagnostic tile'); axes[1].axis('off')
    axes[2].axis('off')
plt.tight_layout(); plt.show()


---
## 7 · Diagnostic — single-tile inference


In [ ]:
_results = batch_forward_v51(
    model, [sample_rgb], DEVICE,
    fp16=USE_FP16, bf16=USE_BF16, channels_last=USE_CHANNELS_LAST,
    cls_perm_gpu=_CLS_PERM_GPU,
)
prob, dist_map, cls_log = _results[0]
cls_argmax = cls_log.argmax(axis=0)
_loose = local_peaks(prob, min_distance=3, thresh=0.30)
peaks = local_peaks(prob, min_distance=int(NMS_DIST), thresh=float(PROB_THRESH))
_, coords_pk = dists_and_coords_from_peaks(
    dist_map, peaks, prob,
    refine_local_com=REFINE_PEAK_LOCAL_COM,
    refine_radius_px=int(REFINE_PEAK_RADIUS),
)
print(f'peaks: loose={len(_loose)}  final={len(peaks)}')

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(sample_rgb); axes[0].set_title('RGB'); axes[0].axis('off')
axes[1].imshow(prob, cmap='hot', vmin=0, vmax=1)
axes[1].set_title(f'Prob (thresh={PROB_THRESH})'); axes[1].axis('off')
_fg_mask = prob > 0.1
_disp = np.where(_fg_mask, cls_argmax, -1).astype(float)
_disp[_disp < 0] = np.nan
axes[2].imshow(_disp, cmap=CMAP_CLASS, norm=NORM_CLASS)
axes[2].set_title('Cls argmax (FG only)'); axes[2].axis('off')
axes[3].imshow(sample_rgb)
for k in range(coords_pk.shape[0]):
    pr, pc = int(peaks[k, 0]), int(peaks[k, 1])
    hw = max(1, 9 // 2)
    r0 = max(0, pr - hw); r1 = min(prob.shape[0], pr + hw)
    c0 = max(0, pc - hw); c1 = min(prob.shape[1], pc + hw)
    lw = cls_log[:, r0:r1, c0:c1].mean(axis=(1, 2))
    cls_id = int(lw.argmax())
    ring = polygon_ring_rowcol(coords_pk[k])
    axes[3].plot(ring[:, 0], ring[:, 1], linewidth=0.7,
                 color=_color_for_class_idx(cls_id))
axes[3].set_title(f'Polygons ({coords_pk.shape[0]} nuclei)')
axes[3].axis('off')
plt.tight_layout(); plt.show()


---
## 8 · Batch inference — all slides in folder

Per slide:
1. **Patch B**: tissue mask via Otsu, with coverage safety net.
2. **Patch E**: auto-pick `LOAD_TO_RAM` based on slide size, available RAM, network share.
3. **Patch B**: bbox-only `LOAD_TO_RAM` (clipped to slide bounds), if RAM mode.
4. **Patch A**: forward via `batch_forward_v51` (bulk D2H, no autocast).
5. **Patch C**: centroid dedup at 12 px; polygon dedup OFF.
6. Streaming GeoJSON write.
7. **Patch D**: optional `SMOKE_TEST_MAX_TILES` for fast iteration.


In [ ]:
if PERMUTE_LEGACY_TO_DISPLAY and cls_perm is None:
    raise RuntimeError('cls_perm is None — re-run Cell 3 before running inference.')

torch.backends.cudnn.benchmark = True

_v51_kwargs = dict(
    fp16=USE_FP16, bf16=USE_BF16, channels_last=USE_CHANNELS_LAST,
    cls_perm_gpu=_CLS_PERM_GPU,
    nms_dist=NMS_DIST, prob_thresh=PROB_THRESH,
    refine_local_com=REFINE_PEAK_LOCAL_COM, refine_radius_px=REFINE_PEAK_RADIUS,
    valid_margin=VALID_MARGIN,
    use_fast_vote_class=True, vote_window_px=9,
    coord_int=GEOJSON_COORD_INT,
    include_class_probs=INCLUDE_CLASS_PROBS,
)

all_wsi_files = sorted(WSI_DIR.glob(WSI_GLOB))
print(f'Found {len(all_wsi_files)} WSI files to process')
all_summaries = []


def _process_one_slide(wsi_idx: int, SLIDE_PATH):
    print(f"\n{'=' * 80}")
    print(f'Processing WSI {wsi_idx + 1}/{len(all_wsi_files)}: {SLIDE_PATH.name}')
    print(f"{'=' * 80}")

    timers = {}
    t_slide = time.perf_counter()
    try:
        slide = openslide.OpenSlide(str(SLIDE_PATH))
    except Exception as e:
        print(f'  ERROR opening slide: {e}')
        return None
    W_s, H_s = slide.level_dimensions[SLIDE_LEVEL]
    print(f'  Dimensions (level {SLIDE_LEVEL}): {W_s} × {H_s}')

    # ── Patch E: auto-pick LOAD_TO_RAM ──────────────────────────────────────
    if AUTO_LOAD_MODE:
        load_to_ram_this, reason = auto_load_to_ram(
            SLIDE_PATH, slide, SLIDE_LEVEL,
            max_gb=AUTO_RAM_MAX_GB, ram_frac=AUTO_RAM_FRAC,
        )
        print(f'  Mode: LOAD_TO_RAM={load_to_ram_this}  ({reason})')
    else:
        load_to_ram_this = LOAD_TO_RAM
        print(f'  Mode: LOAD_TO_RAM={load_to_ram_this}  (manual)')

    # ── Stage 1: tissue mask ─────────────────────────────────────────────────
    _t = time.perf_counter()
    coverage = 0.0
    tissue_mask, mask_scale = None, 1.0
    bx, by, bw, bh = 0, 0, W_s, H_s
    mask_disabled_for_slide = False
    if TISSUE_MASK_ENABLED:
        tissue_mask, mask_scale = load_or_build_tissue_mask(
            slide, SLIDE_PATH, OUT_DIR, cache=TISSUE_MASK_CACHE,
            target_thumbnail_w=TISSUE_MASK_THUMB_W,
            use_otsu=TISSUE_MASK_USE_OTSU,
            val_thresh=TISSUE_MASK_VAL_THRESH,
            sat_thresh=TISSUE_MASK_SAT_THRESH,
            min_component_frac=TISSUE_MASK_MIN_COMP_FRAC,
            dilate_tiles=TISSUE_MASK_DILATE_TILES,
            tile_size_l0=TILE_SIZE,
        )
        coverage = float(tissue_mask.mean())
        # Safety net: if mask covers almost the whole slide, it's not actually filtering.
        if coverage > TISSUE_COVERAGE_HIGH:
            print(f'  WARN: mask covers {coverage:.1%} > {TISSUE_COVERAGE_HIGH:.0%} — disabling ROI filter')
            mask_disabled_for_slide = True
        elif coverage < TISSUE_COVERAGE_LOW:
            print(f'  WARN: mask covers {coverage:.1%} < {TISSUE_COVERAGE_LOW:.0%} — may have lost real tissue')
        if not mask_disabled_for_slide:
            bx, by, bw, bh = tissue_bbox_level0(
                tissue_mask, mask_scale, (W_s, H_s), pad_px=TILE_SIZE,
            )
            if bw == 0 or bh == 0:
                print('  Tissue mask is empty — skipping slide.')
                slide.close()
                return None
    timers['tissue_mask'] = time.perf_counter() - _t

    # ── Stage 2: tile grid + filter ─────────────────────────────────────────
    _t = time.perf_counter()
    tile_coords_all = build_tile_coords(W_s, H_s, TILE_SIZE, TILE_OVERLAP)
    n_total = len(tile_coords_all)
    if TISSUE_MASK_ENABLED and not mask_disabled_for_slide:
        tile_coords, n_dropped = filter_tile_coords_by_mask(
            tile_coords_all, tissue_mask, mask_scale, frac=TISSUE_TILE_KEEP_FRAC,
        )
    else:
        tile_coords, n_dropped = tile_coords_all, 0
    if SMOKE_TEST_MAX_TILES > 0:
        tile_coords = tile_coords[:SMOKE_TEST_MAX_TILES]
        print(f'  SMOKE TEST: limited to {len(tile_coords)} tiles')
    timers['tile_filter'] = time.perf_counter() - _t
    print(f'  Tiles  : {n_total:,} total  →  {len(tile_coords):,} kept  '
          f'({100 * n_dropped / max(n_total, 1):.1f}% skipped, coverage={coverage:.1%})')

    # ── Stage 3: load to RAM (bbox-only) — only if AUTO picked True ─────────
    _t = time.perf_counter()
    _slide_ram = None
    _ram_origin = (0, 0)
    if load_to_ram_this:
        if (TISSUE_MASK_ENABLED and TISSUE_BBOX_CROP
                and not mask_disabled_for_slide and bw > 0 and bh > 0):
            _ram_origin = (bx, by)
            _mb = bw * bh * 3 / 1e6
            print(f'  Loading bbox {bw}×{bh} px to RAM (~{_mb:.0f} MB) …')
            _slide_ram = np.asarray(
                slide.read_region((bx, by), SLIDE_LEVEL, (bw, bh)).convert('RGB'),
                dtype=np.uint8,
            )
        else:
            _mb = W_s * H_s * 3 / 1e6
            print(f'  Loading full slide {W_s}×{H_s} to RAM (~{_mb:.0f} MB) …')
            _slide_ram = np.asarray(
                slide.read_region((0, 0), SLIDE_LEVEL, (W_s, H_s)).convert('RGB'),
                dtype=np.uint8,
            )
    timers['load_to_ram'] = time.perf_counter() - _t
    if load_to_ram_this:
        print(f'    Done in {timers["load_to_ram"]:.1f} s')

    # ── Stage 4: forward + post-processing ──────────────────────────────────
    print(f'  Batch={BATCH_TILES}  workers={NUM_WORKERS}  prefetch={PREFETCH}  '
          f'precision={"bf16" if USE_BF16 else "fp16" if USE_FP16 else "fp32"}'
          + ('  channels_last' if USE_CHANNELS_LAST else ''))

    ALL_FEATURES = []
    t_io = 0.0
    t_gpu = 0.0
    _desc = f'Infer {SLIDE_PATH.stem[:30]}'

    _t_phase = time.perf_counter()
    if _slide_ram is None:
        # Disk-backed DataLoader path
        tile_loader = make_wsi_tile_dataloader(
            str(SLIDE_PATH),
            SLIDE_LEVEL,
            tile_coords,
            batch_size=BATCH_TILES,
            num_workers=NUM_WORKERS,
            prefetch_factor=PREFETCH,
            pin_memory=(DEVICE.type == 'cuda'),
            persistent_workers=True,
        )
        try:
            pbar = tqdm(total=len(tile_coords), desc=_desc, unit='tile')
            for patches, idxs in tile_loader:
                meta = [tile_coords[i] for i in idxs]
                new_feats, _dt = process_tile_batch_v51(
                    meta, patches, model, DEVICE,
                    w_slide=W_s, h_slide=H_s, idx2label=idx2label,
                    color_for_name=_color_for_class_name,
                    **_v51_kwargs,
                )
                t_gpu += _dt
                ALL_FEATURES.extend(new_feats)
                pbar.update(len(meta))
                pbar.set_postfix(nuclei=len(ALL_FEATURES))
            pbar.close()
        finally:
            del tile_loader   # explicit teardown — avoids Windows DataLoader cleanup hang
    else:
        # RAM-backed thread pool path
        with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
            pbar = tqdm(total=len(tile_coords), desc=_desc, unit='tile')
            for bs in range(0, len(tile_coords), BATCH_TILES):
                meta = tile_coords[bs: bs + BATCH_TILES]
                _t = time.perf_counter()
                futs = [
                    pool.submit(
                        get_tile_from_ram,
                        _slide_ram,
                        x0 - _ram_origin[0], y0 - _ram_origin[1], tw, th,
                    )
                    for (x0, y0, tw, th) in meta
                ]
                patches = [f.result() for f in futs]
                t_io += time.perf_counter() - _t
                new_feats, _dt = process_tile_batch_v51(
                    meta, patches, model, DEVICE,
                    w_slide=W_s, h_slide=H_s, idx2label=idx2label,
                    color_for_name=_color_for_class_name,
                    **_v51_kwargs,
                )
                t_gpu += _dt
                ALL_FEATURES.extend(new_feats)
                pbar.update(len(meta))
                pbar.set_postfix(nuclei=len(ALL_FEATURES))
            pbar.close()
    timers['forward+postproc'] = time.perf_counter() - _t_phase
    timers['  of which gpu_forward'] = t_gpu
    timers['  of which tile_io']     = t_io
    n_raw = len(ALL_FEATURES)
    print(f'  Forward+postproc : {timers["forward+postproc"]:.1f} s  '
          f'(GPU {t_gpu:.1f} s, tile I/O {t_io:.1f} s)')
    print(f'  Raw detections : {n_raw:,}')

    # ── Stage 5: dedup ──────────────────────────────────────────────────────
    _t = time.perf_counter()
    ALL_FEATURES = dedupe_nucleus_features_by_centroid(
        ALL_FEATURES, min_dist_px=float(DEDUP_MIN_DIST_PX),
    )
    timers['dedup_centroid'] = time.perf_counter() - _t
    n_after_centroid = len(ALL_FEATURES)
    print(f'  After centroid dedup : {n_after_centroid:,}  ({timers["dedup_centroid"]:.1f} s)')

    if DEDUP_USE_POLYGON_OVERLAP:
        _t = time.perf_counter()
        ALL_FEATURES = dedupe_nucleus_features_by_polygon_overlap(
            ALL_FEATURES,
            min_overlap_ratio=float(DEDUP_OVERLAP_RATIO),
            min_iou=float(DEDUP_MIN_IOU) if DEDUP_MIN_IOU else None,
            grid_cell_px=float(DEDUP_OVERLAP_GRID_PX),
        )
        timers['dedup_polygon'] = time.perf_counter() - _t
        print(f'  After overlap dedup  : {len(ALL_FEATURES):,}  ({timers["dedup_polygon"]:.1f} s)')
    else:
        timers['dedup_polygon'] = 0.0

    n_total_kept = len(ALL_FEATURES)

    # ── Stage 6: GeoJSON write ──────────────────────────────────────────────
    for i, f in enumerate(ALL_FEATURES):
        f['id'] = str(i)
    _gj_suf = '.geojson.gz' if GEOJSON_GZIP else '.geojson'
    path_cls = OUT_DIR / f'{SLIDE_PATH.stem}_classified{_gj_suf}'
    _t = time.perf_counter()
    n_written = write_geojson_streaming(
        ALL_FEATURES, path_cls,
        gzip_compress=GEOJSON_GZIP,
        flush_bytes=GEOJSON_FLUSH_BYTES,
    )
    timers['geojson_write'] = time.perf_counter() - _t
    file_mb = path_cls.stat().st_size / (1024 * 1024)
    print(f'  GeoJSON write : {timers["geojson_write"]:.1f} s  '
          f'({n_written:,} features, {file_mb:.1f} MB)')

    # ── Stage 7: per-class counts + summary ─────────────────────────────────
    class_counts = {}
    for feat in ALL_FEATURES:
        nm = feat['properties']['classification']['name']
        class_counts[nm] = class_counts.get(nm, 0) + 1

    elapsed_total = time.perf_counter() - t_slide
    timers['total'] = elapsed_total

    summary = {
        'wsi_dir': str(WSI_DIR),
        'slide_index': wsi_idx,
        'slide_name': SLIDE_PATH.name,
        'weights': str(WEIGHTS_PATH),
        'slide_level': SLIDE_LEVEL,
        'tile_size': TILE_SIZE, 'tile_overlap': TILE_OVERLAP,
        'valid_margin': VALID_MARGIN,
        'prob_thresh': PROB_THRESH, 'nms_dist': NMS_DIST,
        'precision': 'bf16' if USE_BF16 else 'fp16' if USE_FP16 else 'fp32',
        'channels_last': bool(USE_CHANNELS_LAST),
        'load_to_ram_used': bool(load_to_ram_this),
        'num_workers': NUM_WORKERS, 'prefetch': PREFETCH,
        'tissue_mask_enabled': TISSUE_MASK_ENABLED,
        'tissue_mask_use_otsu': TISSUE_MASK_USE_OTSU,
        'tissue_mask_coverage': round(coverage, 4),
        'tissue_mask_disabled_high_coverage': mask_disabled_for_slide,
        'tiles_total': n_total,
        'tiles_kept': len(tile_coords),
        'tiles_skipped_pct': round(100 * n_dropped / max(n_total, 1), 1),
        'tissue_bbox_level0': [int(bx), int(by), int(bw), int(bh)],
        'raw_detections': n_raw,
        'after_centroid_dedup': n_after_centroid,
        'total_nuclei': n_total_kept,
        'dedup_polygon_used': DEDUP_USE_POLYGON_OVERLAP,
        'class_counts': class_counts,
        'geojson_classified': str(path_cls),
        'geojson_size_mb': round(file_mb, 2),
        'timers_seconds': {k: round(v, 3) for k, v in timers.items()},
    }
    summary_path = OUT_DIR / f'{SLIDE_PATH.stem}_summary.json'
    summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

    print(f'  Summary → {summary_path}')
    print(f"  TOTAL: {elapsed_total:.1f} s ({elapsed_total / 60:.1f} min)")
    print('  Stage breakdown:')
    for k, v in timers.items():
        pct = 100 * v / elapsed_total if elapsed_total > 0 else 0.0
        print(f'    {k:30s} {v:7.2f} s  ({pct:5.1f}%)')
    print('  Top per-class:')
    for nm, cnt in sorted(class_counts.items(), key=lambda x: -x[1])[:8]:
        print(f'    {nm:16s} {cnt:,}')

    slide.close()
    del _slide_ram
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return summary


for wsi_idx, slide_path in enumerate(all_wsi_files):
    if wsi_idx == 0 and RUN_CPROFILE_FIRST_SLIDE:
        print('\n[cProfile] wrapping first slide …')
        pr = cProfile.Profile(); pr.enable()
        s = _process_one_slide(wsi_idx, slide_path)
        pr.disable()
        sio = io.StringIO()
        pstats.Stats(pr, stream=sio).sort_stats('cumulative').print_stats(30)
        print('\n[cProfile] top 30 cumulative time entries:')
        print(sio.getvalue())
    else:
        s = _process_one_slide(wsi_idx, slide_path)
    if s is not None:
        all_summaries.append(s)


---
## 8b · Combined summary across all slides


In [ ]:
print(f"\n{'=' * 80}")
print('ALL WSI PROCESSING COMPLETE')
print(f"{'=' * 80}")

if not all_summaries:
    print('No slides successfully processed.')
else:
    total_nuclei_all = sum(s['total_nuclei'] for s in all_summaries)
    total_time_all = sum(s['timers_seconds']['total'] for s in all_summaries)
    combined_class_counts = {}
    for s in all_summaries:
        for cls_name, cnt in s['class_counts'].items():
            combined_class_counts[cls_name] = combined_class_counts.get(cls_name, 0) + cnt
    stage_totals = {}
    for s in all_summaries:
        for k, v in s['timers_seconds'].items():
            stage_totals[k] = stage_totals.get(k, 0.0) + v

    combined_summary = {
        'total_slides_processed': len(all_summaries),
        'total_nuclei_all_slides': total_nuclei_all,
        'total_time_all_slides_s': round(total_time_all, 2),
        'total_time_all_slides_min': round(total_time_all / 60, 2),
        'avg_time_per_slide_s': round(total_time_all / len(all_summaries), 2),
        'stage_total_seconds': {k: round(v, 2) for k, v in stage_totals.items()},
        'combined_class_counts': combined_class_counts,
        'individual_summaries': all_summaries,
    }
    out = OUT_DIR / 'all_slides_summary.json'
    out.write_text(json.dumps(combined_summary, indent=2), encoding='utf-8')

    print(f'\nProcessed {len(all_summaries)} slides')
    print(f'Total nuclei : {total_nuclei_all:,}')
    print(f'Total time   : {total_time_all:.1f} s ({total_time_all/60:.1f} min)')
    print(f'Avg per slide: {total_time_all/len(all_summaries):.1f} s')
    print('\nStage breakdown across all slides:')
    for k, v in stage_totals.items():
        pct = 100 * v / total_time_all if total_time_all > 0 else 0.0
        print(f'  {k:30s} {v:8.1f} s  ({pct:5.1f}%)')
    print('\nCombined class counts:')
    for nm, cnt in sorted(combined_class_counts.items(), key=lambda x: -x[1]):
        print(f'  {nm:16s} {cnt:,}')
    print(f'\nCombined summary saved to: {out}')


---
## 9 · Results bar chart (last slide)


In [ ]:
if all_summaries:
    last = all_summaries[-1]
    class_counts = last['class_counts']
    sorted_classes = sorted(class_counts.items(), key=lambda x: -x[1])
    names = [x[0] for x in sorted_classes]
    cnts  = [x[1] for x in sorted_classes]
    cols  = [np.array(_color_for_class_name(n)) / 255.0 for n in names]

    fig, ax = plt.subplots(figsize=(10, max(4, len(names) * 0.45)))
    ax.barh(names, cnts, color=cols)
    ax.set_xlabel('Nucleus count')
    ax.invert_yaxis()
    ax.set_title(f"{Path(last['slide_name']).name}  —  {last['total_nuclei']:,} nuclei total")
    for i, v in enumerate(cnts):
        ax.text(v * 1.005, i, f'{v:,}', va='center', fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{Path(last['slide_name']).stem}_class_counts.png", dpi=150)
    plt.show()
else:
    print('No summaries available — run Cell 8 first.')
